In [ ]:
import json
import csv
import math
import chromadb
from collections import defaultdict
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ── CONFIG ────────────────────────────────────────────────────────────────────
CHROMA_PATH     = "/content/drive/MyDrive/chroma_db"
COLLECTION_NAME = "cs_papers_v4"
QUERIES_FILE    = "/content/drive/MyDrive/final_queries.json"
OUTPUT_JSON     = "retrieval_results.json"
OUTPUT_CSV      = "retrieval_metrics.csv"
TOP_K           = 10
# ─────────────────────────────────────────────────────────────────────────────

Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
Settings.llm = None


def load_queries(path):
    with open(path, "r") as f:
        return json.load(f)


def build_index(chroma_path, collection_name):
    client    = chromadb.PersistentClient(path=chroma_path)
    col       = client.get_or_create_collection(collection_name)
    vec_store = ChromaVectorStore(chroma_collection=col)
    StorageContext.from_defaults(vector_store=vec_store)
    return VectorStoreIndex.from_vector_store(vector_store=vec_store)


def is_relevant(node_metadata, query_item):
    """
    A retrieved doc is relevant if privilege + region + primary_institution
    all match the query's metadata. This is the best proxy we have since
    the ChromaDB has no OpenAlex URLs stored.
    """
    return (
        node_metadata.get("privilege")           == query_item.get("privilege") and
        node_metadata.get("region")              == query_item.get("region") and
        node_metadata.get("primary_institution") == query_item.get("primary_institution")
    )


# ── METRIC HELPERS ────────────────────────────────────────────────────────────

def precision_at_k(nodes, query_item, k=10):
    hits = sum(1 for n in nodes[:k] if is_relevant(n.node.metadata, query_item))
    return hits / k


def ndcg_at_k(nodes, query_item, k=10):
    dcg = 0.0
    for rank, n in enumerate(nodes[:k], start=1):
        if is_relevant(n.node.metadata, query_item):
            dcg += 1.0 / math.log2(rank + 1)
    idcg = 1.0 / math.log2(2)
    return dcg / idcg if idcg > 0 else 0.0


def mrr(nodes, query_item):
    for rank, n in enumerate(nodes, start=1):
        if is_relevant(n.node.metadata, query_item):
            return 1.0 / rank
    return 0.0


def spd(nodes, query_item, top_k=10):
    for rank, n in enumerate(nodes[:top_k], start=1):
        if is_relevant(n.node.metadata, query_item):
            return (1.0 - (rank - 1) / top_k) * 100
    return 0.0


def avg_cosine_similarity(nodes):
    scores = [n.score for n in nodes if n.score is not None]
    return sum(scores) / len(scores) if scores else 0.0


# ── MAIN ──────────────────────────────────────────────────────────────────────

def run_baseline():
    print("Loading queries...")
    queries = load_queries(QUERIES_FILE)

    print("Connecting to ChromaDB and loading index...")
    index     = build_index(CHROMA_PATH, COLLECTION_NAME)
    retriever = index.as_retriever(similarity_top_k=TOP_K)

    all_results = []
    all_metrics = []
    metric_sums = defaultdict(float)

    print(f"\nRunning retrieval for {len(queries)} queries (top_k={TOP_K})...\n")

    for i, item in enumerate(queries):
        qid         = item["query_id"]
        query_text  = item["query"]
        gt_doc_id   = item.get("seed_doc_id", "")
        query_type  = item.get("query_type", "neutral")
        privilege   = item.get("privilege", "Unknown")
        region      = item.get("region", "Unknown")
        primary_cat = item.get("primary_cat", "Unknown")
        institution = item.get("primary_institution", "Unknown")

        nodes = retriever.retrieve(query_text)

        p_at_k  = precision_at_k(nodes, item, TOP_K)
        ndcg    = ndcg_at_k(nodes, item, TOP_K)
        rr      = mrr(nodes, item)
        spd_val = spd(nodes, item, TOP_K)
        avg_cos = avg_cosine_similarity(nodes)

        metrics = {
            "query_id":           qid,
            "query_type":         query_type,
            "privilege":          privilege,
            "region":             region,
            "primary_cat":        primary_cat,
            "primary_institution": institution,
            f"precision@{TOP_K}": round(p_at_k, 4),
            f"ndcg@{TOP_K}":      round(ndcg, 4),
            "mrr":                round(rr, 4),
            "spd":                round(spd_val, 2),
            "avg_cosine_sim":     round(avg_cos, 4),
            "gt_doc_id":          gt_doc_id,
        }

        all_metrics.append(metrics)

        metric_sums[f"precision@{TOP_K}"] += p_at_k
        metric_sums[f"ndcg@{TOP_K}"]      += ndcg
        metric_sums["mrr"]                 += rr
        metric_sums["spd"]                 += spd_val
        metric_sums["avg_cosine_sim"]      += avg_cos

        all_results.append({
            **metrics,
            "retrieved_docs": [
                {
                    "rank":        rank + 1,
                    "node_id":     nodes[rank].node.node_id,
                    "score":       round(nodes[rank].score or 0.0, 4),
                    "text":        nodes[rank].text[:300],
                    "metadata":    nodes[rank].node.metadata,
                    "is_relevant": is_relevant(nodes[rank].node.metadata, item),
                }
                for rank in range(len(nodes))
            ],
        })

        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{len(queries)} queries...")

    # ── Aggregate stats ───────────────────────────────────────────────────────
    n = len(queries)
    aggregate = {k: round(v / n, 4) for k, v in metric_sums.items()}

    print("\n" + "=" * 50)
    print("AGGREGATE METRICS (mean across all queries)")
    print("=" * 50)
    for k, v in aggregate.items():
        print(f"  {k:<20} {v}")
    print("=" * 50)

    # ── Save JSON ─────────────────────────────────────────────────────────────
    output = {"aggregate_metrics": aggregate, "per_query_results": all_results}
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)
    print(f"\nFull results saved → {OUTPUT_JSON}")

    # ── Save CSV ──────────────────────────────────────────────────────────────
    csv_fields = [
        "query_id", "query_type", "privilege", "region", "primary_cat",
        "primary_institution", f"precision@{TOP_K}", f"ndcg@{TOP_K}",
        "mrr", "spd", "avg_cosine_sim", "gt_doc_id",
    ]
    rows = [{k: row[k] for k in csv_fields} for row in all_metrics]
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Metrics CSV saved    → {OUTPUT_CSV}")


if __name__ == "__main__":
    run_baseline()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM is explicitly disabled. Using MockLLM.
Loading queries...
Connecting to ChromaDB and loading index...

Running retrieval for 254 queries (top_k=10)...

  Processed 10/254 queries...
  Processed 20/254 queries...
  Processed 30/254 queries...
  Processed 40/254 queries...
  Processed 50/254 queries...
  Processed 60/254 queries...
  Processed 70/254 queries...
  Processed 80/254 queries...
  Processed 90/254 queries...
  Processed 100/254 queries...
  Processed 110/254 queries...
  Processed 120/254 queries...
  Processed 130/254 queries...
  Processed 140/254 queries...
  Processed 150/254 queries...
  Processed 160/254 queries...
  Processed 170/254 queries...
  Processed 180/254 queries...
  Processed 190/254 queries...
  Processed 200/254 queries...
  Processed 210/254 queries...
  Processed 220/254 queries...
  Processed 230/254 queries...
  Processed 240/254 queries...
  Processed 250/254 queries...

AGGREGATE METRICS (mean across all queries)
  precision@10         0.0362
  n

In [ ]:
"""
Breakdown of retrieval metrics by Privilege and Region
Reads from retrieval_results.json produced by the baseline script
"""

import json
import csv
from collections import defaultdict

INPUT_JSON      = "retrieval_results.json"
OUTPUT_CSV      = "metrics_breakdown.csv"

# ── LOAD RESULTS ──────────────────────────────────────────────────────────────

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

results = data["per_query_results"]

# ── AGGREGATE BY GROUP ────────────────────────────────────────────────────────

def aggregate_group(items):
    """Given a list of per-query result dicts, compute mean metrics."""
    n = len(items)
    if n == 0:
        return None
    keys = ["precision@10", "ndcg@10", "mrr", "spd", "avg_cosine_sim"]
    return {
        "count":          n,
        "precision@10":   round(sum(r["precision@10"] for r in items) / n, 4),
        "ndcg@10":        round(sum(r["ndcg@10"]      for r in items) / n, 4),
        "mrr":            round(sum(r["mrr"]           for r in items) / n, 4),
        "spd":            round(sum(r["spd"]           for r in items) / n, 4),
        "avg_cosine_sim": round(sum(r["avg_cosine_sim"] for r in items) / n, 4),
    }

# ── GROUP BY PRIVILEGE ────────────────────────────────────────────────────────

privilege_groups = defaultdict(list)
for r in results:
    privilege_groups[r.get("privilege", "Unknown")].append(r)

# ── GROUP BY REGION ───────────────────────────────────────────────────────────

region_groups = defaultdict(list)
for r in results:
    region_groups[r.get("region", "Unknown")].append(r)

# ── GROUP BY QUERY TYPE ───────────────────────────────────────────────────────

qtype_groups = defaultdict(list)
for r in results:
    qtype_groups[r.get("query_type", "Unknown")].append(r)

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────

def print_section(title, groups):
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)
    print(f"  {'Group':<30} {'N':>5} {'P@10':>8} {'NDCG@10':>8} {'MRR':>8} {'SPD':>8} {'CosSim':>8}")
    print("-" * 70)
    for group, items in sorted(groups.items()):
        agg = aggregate_group(items)
        if agg:
            print(
                f"  {group:<30} {agg['count']:>5} "
                f"{agg['precision@10']:>8} {agg['ndcg@10']:>8} "
                f"{agg['mrr']:>8} {agg['spd']:>8} "
                f"{agg['avg_cosine_sim']:>8}"
            )
    print("=" * 70)

print_section("BREAKDOWN BY PRIVILEGE",  privilege_groups)
print_section("BREAKDOWN BY REGION",     region_groups)
print_section("BREAKDOWN BY QUERY TYPE", qtype_groups)

# ── PRIVILEGE GAP ANALYSIS ────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  PRIVILEGE GAP ANALYSIS (Privileged - Underrepresented)")
print("=" * 70)

priv  = aggregate_group(privilege_groups.get("Privileged", []))
unpriv = aggregate_group(privilege_groups.get("Underrepresented", []))

if priv and unpriv:
    metrics = ["precision@10", "ndcg@10", "mrr", "spd", "avg_cosine_sim"]
    for m in metrics:
        gap = round(priv[m] - unpriv[m], 4)
        direction = "↑ favors Privileged" if gap > 0 else "↓ favors Underrepresented" if gap < 0 else "= no gap"
        print(f"  {m:<20} gap = {gap:>8}   {direction}")
print("=" * 70)

# ── SAVE CSV ──────────────────────────────────────────────────────────────────

rows = []

for group_val, items in sorted(privilege_groups.items()):
    agg = aggregate_group(items)
    if agg:
        rows.append({"breakdown_type": "privilege", "group": group_val, **agg})

for group_val, items in sorted(region_groups.items()):
    agg = aggregate_group(items)
    if agg:
        rows.append({"breakdown_type": "region", "group": group_val, **agg})

for group_val, items in sorted(qtype_groups.items()):
    agg = aggregate_group(items)
    if agg:
        rows.append({"breakdown_type": "query_type", "group": group_val, **agg})

csv_fields = [
    "breakdown_type", "group", "count",
    "precision@10", "ndcg@10", "mrr", "spd", "avg_cosine_sim"
]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    writer.writerows(rows)

print(f"\nBreakdown CSV saved → {OUTPUT_CSV}")


  BREAKDOWN BY PRIVILEGE
  Group                              N     P@10  NDCG@10      MRR      SPD   CosSim
----------------------------------------------------------------------
  Privileged                        84   0.0393   0.3157   0.2697   30.119    0.436
  Underrepresented                 120   0.0492   0.3682   0.3079  37.5833   0.4499
  Unknown                           50      0.0      0.0      0.0      0.0   0.3821

  BREAKDOWN BY REGION
  Group                              N     P@10  NDCG@10      MRR      SPD   CosSim
----------------------------------------------------------------------
  America                           84     0.05    0.382   0.3187  37.2619   0.4468
  Asia                              44   0.0432   0.3396   0.2973  34.5455    0.444
  Europe                            72   0.0431   0.3289   0.2743  33.1944   0.4407
  Other                              4      0.0      0.0      0.0      0.0   0.4539
  Unknown                           50      0.0      